# peekabook-crs-test4 결과 분석

wandb 프로젝트 `peekabook-crs-test4`에서 실험 결과를 불러와
NDCG@3,5,10 / Hit-rate@3,5,10 / 평균 점수 / 평균 비용을 계산

**전제**: 각 런에 `book_1_score` ~ `book_10_score` (0 또는 1) 가 기록되어 있어야 함

In [1]:
import math
import pandas as pd
import wandb

# ── 설정 ─────────────────────────────────────────────────────────────────────
WANDB_PROJECT  = "jjeong3150-aiffel/peekabook-crs-test4"
N_BOOKS        = 10   # 리랭킹 후 평가 대상 권수

# 비용 계산용 단가 (USD per 1M tokens) — 사용 모델에 맞게 수정
MODEL_COSTS = {
    "gpt-4o-mini":       (0.15,  0.60),
    "gpt-4o":            (2.50, 10.00)
}
ACTIVE_MODEL = "gpt-4o-mini"

In [2]:
# ── 지표 계산 함수 ─────────────────────────────────────────────────────────

def dcg_at_k(scores: list, k: int) -> float:
    return sum(r / math.log2(i + 2) for i, r in enumerate(scores[:k]))

def ndcg_at_k(scores: list, k: int) -> float:
    dcg  = dcg_at_k(scores, k)
    idcg = dcg_at_k(sorted(scores, reverse=True), k)
    return round(dcg / idcg, 4) if idcg > 0 else 0.0

def hit_rate_at_k(scores: list, k: int) -> int:
    return 1 if any(s > 0 for s in scores[:k]) else 0

def compute_metrics(ranked_scores: list) -> dict:
    """
    ranked_scores: rank 오름차순으로 정렬된 0/1 score 리스트
                   예) rank1→0, rank2→1, rank3→0 이면 [0, 1, 0]
    mean_score@N은 상위 N개 중 1의 비율 (정렬 없이 앞에서 N개)
    """
    n = len(ranked_scores)
    return {
        "ndcg@3":        ndcg_at_k(ranked_scores, 3),
        "ndcg@5":        ndcg_at_k(ranked_scores, 5),
        "ndcg@10":       ndcg_at_k(ranked_scores, 10),
        "hr@3":          hit_rate_at_k(ranked_scores, 3),
        "hr@5":          hit_rate_at_k(ranked_scores, 5),
        "hr@10":         hit_rate_at_k(ranked_scores, 10),
        "mean_score@3":  round(sum(ranked_scores[:3])  / 3,  4) if n >= 3  else 0.0,
        "mean_score@5":  round(sum(ranked_scores[:5])  / 5,  4) if n >= 5  else 0.0,
        "mean_score@10": round(sum(ranked_scores[:10]) / 10, 4) if n >= 10 else 0.0,
    }

def tokens_to_cost(input_tokens: int, output_tokens: int, model: str = ACTIVE_MODEL) -> float:
    in_rate, out_rate = MODEL_COSTS.get(model, MODEL_COSTS["gpt-4o-mini"])
    return (input_tokens * in_rate + output_tokens * out_rate) / 1_000_000

In [3]:
# ── wandb 런 로드 ──────────────────────────────────────────────────────────

api  = wandb.Api()
runs = api.runs(WANDB_PROJECT)

records = []
for run in runs:
    s   = run.summary
    cfg = run.config

    # book_1_score ~ book_N_score 수집
    scores = []
    for i in range(1, N_BOOKS + 1):
        v = s.get(f"book_{i}_score")
        if v is None:
            break
        scores.append(int(v))

    if not scores:
        continue

    cost = tokens_to_cost(
        int(s.get("total_input_tokens",  0)),
        int(s.get("total_output_tokens", 0)),
    )

    records.append({
        "run_id":            run.id,
        "run_name":          run.name,
        "persona_name":      cfg.get("persona_name",     s.get("persona_name", "")),
        "query_transform":   cfg.get("query_transform",  ""),
        "use_genre_filter":  cfg.get("use_genre_filter", ""),
        "collection_name":   cfg.get("collection_name",  ""),
        "session_id":        s.get("session_id", ""),
        "n_books_evaluated": len(scores),
        **compute_metrics(scores),
        "total_input_tokens":  int(s.get("total_input_tokens",  0)),
        "total_output_tokens": int(s.get("total_output_tokens", 0)),
        "estimated_cost_usd":  round(cost, 6),
    })

df = pd.DataFrame(records)
print(f"로드된 런: {len(df)}개")
df.head()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/jjeong3150/.netrc.


로드된 런: 180개


,run_id,run_name,persona_name,query_transform,use_genre_filter,collection_name,session_id,n_books_evaluated,ndcg@3,ndcg@5,ndcg@10,hr@3,hr@5,hr@10,mean_score@3,mean_score@5,mean_score@10,total_input_tokens,total_output_tokens,estimated_cost_usd
0,clvyzegc,comic-sweep-1,A_최재원,none,True,books_intro_48k,115eb735,10,1.0000,0.7227,0.9392,1,1,1,1.0000,0.6,0.6,13714,2268,0.003418
1,90ehfkcx,iconic-sweep-2,A_최재원,none,False,books_intro_48k,d7a9f582,10,1.0000,0.7227,0.9312,1,1,1,1.0000,0.6,0.6,14543,2657,0.003776
2,6q0wwhm6,sage-sweep-3,A_최재원,none,True,books_intro_48k,d652a4d1,10,0.7654,0.6367,0.8670,1,1,1,0.6667,0.4,0.4,11790,1904,0.002911
3,0mcswv72,neat-sweep-4,A_최재원,none,False,books_intro_48k,6792b4b3,10,0.7654,0.5531,0.8753,1,1,1,0.6667,0.4,0.6,14140,2083,0.003371
4,2fyvii3w,mild-sweep-5,A_최재원,none,True,books_intro_48k,99b6487d,10,0.7654,0.5531,0.8684,1,1,1,0.6667,0.4,0.6,13704,2234,0.003396


In [4]:
# ── 조건별 집계 ────────────────────────────────────────────────────────────

GROUP_COLS = ["collection_name", "use_genre_filter"]

summary = (
    df.groupby(GROUP_COLS)
    .agg(
        runs              = ("run_id",              "count"),
        ndcg_3            = ("ndcg@3",              "mean"),
        ndcg_5            = ("ndcg@5",              "mean"),
        ndcg_10           = ("ndcg@10",             "mean"),
        hr_3              = ("hr@3",                "mean"),
        hr_5              = ("hr@5",                "mean"),
        hr_10             = ("hr@10",               "mean"),
        mean_score_top3   = ("mean_score@3",        "mean"),
        mean_score_top5   = ("mean_score@5",        "mean"),
        mean_score_top10  = ("mean_score@10",       "mean"),
        avg_cost_usd      = ("estimated_cost_usd",  "mean"),
    )
    .round(4)
    .reset_index()
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
summary

,collection_name,use_genre_filter,runs,ndcg_3,ndcg_5,ndcg_10,hr_3,hr_5,hr_10,mean_score_top3,mean_score_top5,mean_score_top10,avg_cost_usd
0,books_intro_48k,False,45,0.6426,0.6291,0.8297,0.9778,1.0000,1.0000,0.6148,0.5289,0.4756,0.0029
1,books_intro_48k,True,45,0.5819,0.6007,0.7750,0.8889,0.9333,0.9556,0.5704,0.5111,0.4444,0.0029
2,books_merged_48k,False,45,0.5508,0.5981,0.7608,0.9333,0.9778,1.0000,0.5333,0.4889,0.3978,0.0028
3,books_merged_48k,True,45,0.5074,0.5596,0.7446,0.8889,0.9556,0.9778,0.4741,0.4400,0.3911,0.0029


In [5]:
from scipy import stats

METRIC = "ndcg@3"  # 비교할 지표

# ── 비교 그룹 분리 ─────────────────────────────────────────────────────────

# 1) collection 비교 (intro vs merged), genre_filter 조건 고정
for genre_filter in [True, False]:
    subset = df[df["use_genre_filter"] == genre_filter]
    a = subset[subset["collection_name"] == "books_intro_48k"][METRIC]
    b = subset[subset["collection_name"] == "books_merged_48k"][METRIC]

    t, p = stats.ttest_ind(a, b)
    print(f"[collection 비교 | genre_filter={genre_filter}]")
    print(f"  intro: mean={a.mean():.4f}, n={len(a)}")
    print(f"  merged: mean={b.mean():.4f}, n={len(b)}")
    print(f"  t={t:.4f}, p={p:.4f} {'✅ 유의' if p < 0.05 else '❌ 비유의'}\n")

# 2) genre_filter 비교 (True vs False), collection 조건 고정
for collection in ["books_intro_48k", "books_merged_48k"]:
    subset = df[df["collection_name"] == collection]
    a = subset[subset["use_genre_filter"] == False][METRIC]
    b = subset[subset["use_genre_filter"] == True][METRIC]

    t, p = stats.ttest_ind(a, b)
    print(f"[genre_filter 비교 | collection={collection}]")
    print(f"  filter=False: mean={a.mean():.4f}, n={len(a)}")
    print(f"  filter=True:  mean={b.mean():.4f}, n={len(b)}")
    print(f"  t={t:.4f}, p={p:.4f} {'✅ 유의' if p < 0.05 else '❌ 비유의'}\n")

[collection 비교 | genre_filter=True]
  intro: mean=0.5819, n=45
  merged: mean=0.5074, n=45
  t=1.2659, p=0.2089 ❌ 비유의

[collection 비교 | genre_filter=False]
  intro: mean=0.6426, n=45
  merged: mean=0.5508, n=45
  t=1.7656, p=0.0809 ❌ 비유의

[genre_filter 비교 | collection=books_intro_48k]
  filter=False: mean=0.6426, n=45
  filter=True:  mean=0.5819, n=45
  t=1.0642, p=0.2902 ❌ 비유의

[genre_filter 비교 | collection=books_merged_48k]
  filter=False: mean=0.5508, n=45
  filter=True:  mean=0.5074, n=45
  t=0.8045, p=0.4233 ❌ 비유의



In [6]:
# # ── 페르소나별 평균 ────────────────────────────────────────────────────────

# (
#     df.groupby("persona_name")[metric_cols]
#     .mean()
#     .round(4)
# )

In [7]:
# # ── 런별 상세 (정렬: ndcg@10 내림차순) ────────────────────────────────────

# detail_cols = ["persona_name", "query_transform", "use_genre_filter",
#                "ndcg@3", "ndcg@5", "ndcg@10", "hr@3", "hr@5", "hr@10",
#                "mean_score", "estimated_cost_usd", "session_id"]
# df[detail_cols].sort_values("ndcg@10", ascending=False).reset_index(drop=True)